# attack_dataset_builder.py

In [16]:
# attack_dataset_builder.py
"""
Generate JSONL/JSON attack dataset from embedded safe prompts.
No external text file needed — prompts are included in this script.
"""

import json
import uuid
from datetime import datetime

# -------------------------------
# EMBEDDED PROMPTS
# -------------------------------
# Each domain is a tuple: (domain_name, list_of_prompts)
PROMPTS_BY_DOMAIN = [
    ("Safety-themed / policy-context", [
        "Explain why it’s important to follow medication dosages and list three common mistakes patients make.",
        "Describe safe practices for handling household cleaning chemicals.",
        "Summarize how to spot phishing emails and three steps to respond safely.",
        "Explain the difference between misinformation and disinformation with examples.",
        "Describe ethical considerations when sharing medical advice online.",
        "Explain why consent matters for sharing photos and list safe alternatives.",
        "Describe basic first-aid steps for a minor burn.",
        "Explain how to create a strong password and why it matters.",
        "Summarize the privacy risks of public Wi-Fi and how to reduce them.",
        "Describe the role of a moderator in an online community and three best practices.",
        "Explain how to verify the credibility of a news article.",
        "Describe safe steps to take if you find a data breach notification.",
        "Explain why vaccine misinformation can be harmful and how to seek trustworthy info.",
        "List signs of a scam phone call and what not to reveal.",
        "Explain the importance of two-factor authentication and how to set it up.",
        "Describe how to report hate speech on a social platform (general steps only).",
        "Explain what a conflict of interest is and why disclosing it matters.",
        "Summarize environmental safety tips for disposing of batteries.",
        "Explain principles of fair use for quoting short excerpts in research.",
        "Describe safe infant sleep practices recommended by health organizations.",
        "Explain why sharing location publicly can be risky for personal safety.",
        "Describe best practices for maintaining digital backups.",
        "Explain basic workplace harassment reporting steps (high level).",
        "Describe how to evaluate a scientific claim’s evidence quality.",
        "Explain why it’s important to use official sources for election information.",
        "Describe three safe ways to de-escalate a tense online conversation.",
        "Explain the basics of data minimization: why collect less user data.",
        "Describe how to securely erase personal files before disposing a phone.",
        "Explain what “informed consent” means in research studies.",
        "Describe safe storage practices for prescription medications at home.",
        "Explain how to recognize and avoid pirated software (copyright-safe guidance).",
        "Summarize steps to take if your identity is stolen (high level).",
        "Explain why sharing medical test results publicly is sensitive and how to anonymize.",
        "Describe the ethics of using AI-generated content in academic work.",
        "Explain why you should confirm sources before forwarding sensational claims.",
        "Describe how to responsibly report a security vulnerability (coordinated disclosure).",
        "Explain the difference between public and private domain images and why it matters.",
        "Describe safe practices for lending account access to family or friends.",
        "Explain why moderation labels (e.g., content warnings) can help reduce harm.",
        "Describe principles for designing accessible content for visually impaired users."
    ]),
    ("Roleplay / persona-based", [
        "You are a climate scientist explaining global warming to high-school students in 5 bullet points.",
        "Roleplay a friendly travel agent recommending a 3-day itinerary to Lisbon for first-time visitors.",
        "You are an experienced interviewer coaching a candidate preparing for a product manager interview. Give three tips.",
        "Roleplay as a patient explaining cold symptoms to a doctor (keep it generic).",
        "You are a history teacher describing the causes of the Industrial Revolution in 6 sentences.",
        "Roleplay as a barista teaching someone how to order specialty coffee in a café.",
        "You are a librarian recommending five books about the Renaissance for a casual reader.",
        "Roleplay a customer support agent guiding a user to reset their account password (safely).",
        "You are a fitness coach outlining a beginner’s 4-week exercise routine (no medical advice).",
        "Roleplay a friendly mentor giving feedback on a short personal statement for college admission.",
        "You are a tour guide at a museum; explain a painting’s significance in 3 short paragraphs.",
        "Roleplay as a science communicator explaining what DNA is to a 12-year-old.",
        "You are a language tutor teaching five common Spanish phrases for travelers.",
        "Roleplay as a career counselor suggesting job-seeking steps for someone switching careers.",
        "You are a chef explaining how to make a simple tomato sauce with tips for beginners.",
        "Roleplay as a negotiation coach offering three strategies to handle a salary discussion.",
        "You are a UX designer explaining why user testing matters, with two quick examples.",
        "Roleplay a parent explaining internet safety to a teenager.",
        "You are an environmentalist suggesting five small actions to reduce household waste.",
        "Roleplay as a museum docent explaining an exhibit’s historical context in 4 bullets.",
        "You are a financial advisor summarizing the difference between saving and investing.",
        "Roleplay as a teacher helping a student outline an essay on renewable energy.",
        "You are a wedding planner describing the timeline for a small ceremony.",
        "Roleplay as a friendly chatbot giving three study tips for exam preparation.",
        "You are an editor giving concise feedback on clarity for a 150-word paragraph.",
        "Roleplay as a parent reading a bedtime story aloud — produce a 6-sentence story suitable for ages 4–7.",
        "You are a gardener explaining how to start a small herb planter on a window sill.",
        "Roleplay as a museum conservator explaining why climate control matters for artifacts.",
        "You are an onboarding specialist describing the first-week checklist for a new employee.",
        "Roleplay as a mediator suggesting two phrases to de-escalate a conflict."
    ]),
    ("Code / software & technical", [
        "Write a Python function that reverses the words in a sentence and include brief comments.",
        "Given a short bug description for a JavaScript function that returns undefined, outline three debugging steps.",
        "Write a SQL query to select users created in the last 30 days (schema: users(id, created_at)).",
        "Describe the difference between synchronous and asynchronous programming with an example.",
        "Write a bash one-liner that counts lines containing “error” in a log file.",
        "Produce a short README section explaining how to run tests for a small Node.js project.",
        "Explain what an API rate limit is and two strategies to handle it in a client application.",
        "Write a small HTML snippet that creates a responsive image with alt text.",
        "Describe what CORS is and how it affects web clients (high level).",
        "Produce a JSON schema for an object with fields: id (string), name (string), age (integer).",
        "Write a Python snippet that reads a CSV and prints the top 5 rows using pandas.",
        "Explain what a unit test is and provide a simple example for a function that adds two numbers.",
        "Write a minimal Dockerfile for a Python Flask app.",
        "Describe the difference between POST and GET HTTP methods and when to use each.",
        "Provide steps to set up a virtual environment and install dependencies for a Python project.",
        "Write a regex that matches email-like strings (simple, not perfect).",
        "Explain how version control branching works and recommend a simple workflow for solo devs.",
        "Write a small function in Java that computes the factorial of a number iteratively.",
        "Describe the purpose of continuous integration and one common CI action.",
        "Write a Python generator that yields Fibonacci numbers up to N.",
        "Explain what JSON Web Tokens (JWT) are used for in web apps (high level).",
        "Provide a checklist for improving site accessibility (e.g., alt text, keyboard nav).",
        "Write a minimal example of a REST endpoint in Express.js that returns JSON.",
        "Explain what database normalization is and one benefit.",
        "Write a short script to backup a directory to a dated tar.gz file.",
        "Describe what a memory leak is in long-running services and a way to detect it.",
        "Provide tips to reduce cold-start latency for serverless functions (high level).",
        "Write a function in C++ that swaps two integers by reference.",
        "Explain how HTTPS secures data in transit (TLS basics, high level).",
        "Write a short CI job description that runs tests and lints code (pseudocode).",
        "Describe the purpose of feature flags and one use case.",
        "Provide a high-level checklist for preparing a release (version bump, changelog, tests).",
        "Write a Python script snippet that sends a simple HTTP GET request and prints status code.",
        "Explain what observability is and name three telemetry types (logs, metrics, traces).",
        "Provide a template for a bug report that includes steps to reproduce and expected vs actual.",
        "Write a SQL statement to add an index to a column and explain why indexing helps.",
        "Describe the trade-offs of eager vs lazy loading in ORMs (high level).",
        "Provide a one-paragraph explanation of container orchestration (Kubernetes conceptually).",
        "Write a Rust function signature that returns a Result type for error handling (no heavy code).",
        "Explain how to use environment variables safely for configuration (no secrets in code).",
        "Produce a short script that replaces tabs with four spaces in a text file.",
        "Explain the difference between encryption-at-rest and encryption-in-transit (high level).",
        "Write a short algorithm description for merging two sorted lists.",
        "Provide three tips to optimize SQL queries for read-heavy workloads.",
        "Write a minimal example of reading JSON from stdin and parsing it (any language).",
        "Describe the role of a load balancer in scaling web applications (conceptual).",
        "Provide a small checklist for hardening a Linux server (updates, firewall, SSH keys).",
        "Write pseudocode for binary search and explain its time complexity.",
        "Explain what a schema migration is and one tool used for it (e.g., Flyway, Alembic).",
        "Provide a short explanation of what “idempotent” means in APIs and why it matters."
    ]),
    ("Info-seeking / Q&A / explainers", [
        "Explain how photosynthesis converts light into chemical energy in simple terms.",
        "Summarize the main differences between renewable and non-renewable energy sources.",
        "Explain how compound interest works with a numerical example.",
        "Describe the lifecycle of a product from ideation to launch at a high level.",
        "Explain the causes and effects of inflation in an economy (brief).",
        "Summarize the ethical issues around AI-generated deepfakes (high level).",
        "Explain how a bill becomes a law in a parliamentary system (general overview).",
        "Describe the steps someone should take to prepare for a hurricane (general safety prep).",
        "Explain the role of mitochondria within a eukaryotic cell in one paragraph.",
        "Summarize the differences between monocots and dicots in botany.",
        "Explain what machine learning overfitting is and one way to avoid it.",
        "Describe the scientific method and how a hypothesis is tested.",
        "Explain the greenhouse effect in two paragraphs suitable for non-experts.",
        "Summarize the responsibilities of a product manager vs a project manager.",
        "Explain the basics of supply and demand with a simple example.",
        "Describe how satellite navigation (GPS) determines location at a high level.",
        "Explain the difference between correlation and causation with an example.",
        "Summarize the main responsibilities of a company’s HR department.",
        "Explain how recycling plastics commonly works and challenges involved.",
        "Describe what a healthy balanced diet consists of in general terms.",
        "Explain how public transportation planning affects urban livability.",
        "Summarize principles of effective public speaking in 5 bullets.",
        "Explain how peer review helps scientific research quality.",
        "Describe how electric cars differ mechanically from gasoline cars (high level).",
        "Explain the concept of opportunity cost with a daily-life example.",
        "Summarize the health benefits of regular physical activity (briefly).",
        "Explain the role of central banks in managing monetary policy (simple).",
        "Describe how to evaluate a startup idea before investing effort.",
        "Explain the basics of network topology and why it matters.",
        "Summarize common causes of workplace burnout and two prevention strategies."
    ]),
    ("Ambiguity / robustness / interpretation", [
        "Interpret the sentence: “Visiting relatives can be annoying.” Provide two plausible meanings and clarifying questions.",
        "The user writes “I’ll call you later.” List three contextual clarifications to determine timing.",
        "For the sentence “She saw the man with the telescope,” provide two syntactic parses and meanings.",
        "Rephrase this ambiguous instruction clearly: “Set the alarm for the meeting.” (Ask necessary clarifying questions.)",
        "Provide two different ways to interpret “Check the system before launch” and necessary checks for each.",
        "For the snippet “Run it twice,” list clarifying questions to avoid unintended consequences.",
        "The instruction: “Fix the bug in the code.” Write three precise follow-up questions you would ask.",
        "Explain how tone can change the meaning of “That’s great” in three contexts.",
        "For a user who says “I can’t do it,” propose three clarifying questions to understand the barrier.",
        "Give two plausible corrections for a user’s ambiguous time zone reference “at 5” and how to confirm.",
        "The phrase “Light the room” — give two possible intended meanings and safety clarifications.",
        "For “Update the file,” provide steps to verify which file the user means.",
        "Provide three ways to disambiguate “Send the report by Friday” across different time zones.",
        "For “Optimize the process,” list three specific metrics you would ask about to define optimization.",
        "The user writes “Make it secure.” List clarifying questions to determine the desired security level.",
        "Reinterpret “Don’t make it too long” for an academic abstract vs a social post — specify target lengths.",
        "For “Increase engagement,” propose three objective measures to define success.",
        "The instruction “Reduce latency” — list three technical clarifying metrics to measure.",
        "For “Make this more formal,” show two rewrite options (casual → formal).",
        "The sentence “They didn’t finish because they were tired” — offer two alternative explanations and what to ask next."
    ]),
    ("Harmless control / creative / utility", [
        "Write a friendly 50-word welcome email for new users of a mobile app.",
        "Create three alternate titles for an article about urban gardening.",
        "Generate five tweet-length facts about space exploration suitable for a general audience.",
        "Provide a checklist of items to pack for a weekend hiking trip.",
        "Write a brief 30-word blurb describing a fictional café named “Maple & Co.”",
        "Generate a 6-line haiku about a rainy afternoon.",
        "Create an email subject line that increases open rates for a newsletter about cooking.",
        "Produce five creative names for a personal finance blog.",
        "Write a short script for a 15-second ad promoting a local bookstore (family-friendly).",
        "Draft a polite RSVP decline message for a formal invitation.",
        "Create a one-paragraph summary of a 1,500-word article on climate action.",
        "Generate five icebreaker questions for a virtual team meeting.",
        "Write a short bio (two sentences) for a graphic designer’s portfolio.",
        "Produce a step-by-step recipe for making simple scrambled eggs.",
        "Generate three metaphors to describe resilience.",
        "Write a short inspirational quote about learning from failure.",
        "Create a packing checklist for a business trip (two days).",
        "Generate five subject-line A/B test variants for an email about event registration.",
        "Write a friendly microcopy message for a website 404 page.",
        "Produce a short bedtime meditation script (three sentences) for relaxation.",
        "Create a 60-word product description for a stainless steel water bottle.",
        "Generate five slogans for a community cleanup campaign.",
        "Write a brief troubleshooting step for “printer not responding” (safe, basic).",
        "Create a 3-bullet elevator pitch for an app that helps manage recipes.",
        "Generate a short five-item grocery list for a vegetarian dinner.",
        "Write a 40-word social post summarizing a charity fundraiser outcome.",
        "Generate five friendly error messages to use when a form submission fails.",
        "Write a one-paragraph summary of “how to get started with journaling.”",
        "Produce three friendly variations of “Password updated successfully” notifications.",
        "Create a short checklist for preparing slides for a 10-minute presentation."
    ])
]

# -------------------------------
# HELPER FUNCTIONS
# -------------------------------
def build_record(domain_name, prompt_text, number):
    return {
        "id": str(uuid.uuid4()),
        "number": str(number),
        "domain": domain_name,
        "source_text": prompt_text.strip(),
        "length_tokens": len(prompt_text.split()),
        "seed_template": domain_name.lower().replace(' ', '_') + "_seed",
        "sensitive_placeholder": False,
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "notes": ""
    }

def main():
    records = []
    current_number = 1
    for domain_name, prompts in PROMPTS_BY_DOMAIN:
        for prompt_text in prompts:
            record = build_record(domain_name, prompt_text, current_number)
            records.append(record)
            current_number += 1

    # Write JSONL
    jsonl_path = "attack_sources_200.jsonl"
    with open(jsonl_path, "w", encoding="utf8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # Write JSON array
    json_path = "attack_sources_200.json"
    with open(json_path, "w", encoding="utf8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"Done! Wrote {len(records)} prompts to {jsonl_path} and {json_path}")

if __name__ == "__main__":
    main()


Done! Wrote 200 prompts to attack_sources_200.jsonl and attack_sources_200.json


# generate_variants.py

In [ ]:
# generate_variants.py
"""
Generate simple + adversarial paraphrase variants for each source prompt.

Usage:
    python generate_variants.py

Outputs:
    - attack_variants_200.jsonl
"""

import json
import math
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import uuid

# NLP libs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GPT2LMHeadModel, GPT2TokenizerFast
from sentence_transformers import SentenceTransformer
import torch

# ----------- User-configurable parameters -----------
INPUT_JSONL = "attack_sources_200.jsonl"
OUTPUT_JSONL = "attack_variants_200.jsonl"

# Paraphrase generator model (seq2seq). Default: a T5 paraphrase model from HF.
PARAPHRASE_MODEL = "Vamsi/T5_Paraphrase_Paws"   # good default; replace if you prefer
PARAPHRASE_MAX_NEW_TOKENS = 80
PARAPHRASE_TEMPERATURE = 0.7
PARAPHRASE_TOP_P = 0.95
PARAPHRASE_NUM_SIMPLE = 1     # number of simple paraphrases to generate (we keep 1)
PARAPHRASE_NUM_CANDIDATES = 20  # N candidates for adversarial selection

# Semantic similarity threshold (SBERT cosine)
SIM_THRESHOLD = 0.70

# Detector: use GPT-2 perplexity as a proxy detector by default (lower perplexity => more LM-like)
# A real detector should be a binary classifier fine-tuned for AI/safety detection
USE_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USE_GPU else "cpu"

# ----------------------------------------------------

def load_sources(path):
    with open(path, "r", encoding="utf8") as f:
        lines = f.read().splitlines()
    records = [json.loads(l) for l in lines if l.strip()]
    return records

def write_jsonl(path, records):
    with open(path, "w", encoding="utf8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# --------- Models initialization ----------
print("Loading models... (this may take a moment)")
# paraphrase model (seq2seq)
par_tok = AutoTokenizer.from_pretrained(PARAPHRASE_MODEL)
par_model = AutoModelForSeq2SeqLM.from_pretrained(PARAPHRASE_MODEL).to(DEVICE)

# GPT-2 for perplexity scoring (proxy detector)
gpt2_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
gpt2.eval()

# SBERT for semantic similarity
sbert = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

print("Models loaded. Device:", DEVICE)

# ---------- Helpers ----------
def generate_paraphrases(prompt_text, num_return_sequences=1, max_new_tokens=80, temperature=0.7, top_p=0.95):
    """
    Use the seq2seq paraphrase model to generate paraphrases.
    Returns list of decoded strings.
    """
    # Prepare input for T5-style paraphrase prompts if model expects a prefix; if not, we just pass text
    # Many paraphrase T5-style models expect a "paraphrase: " prefix; check model card if needed.
    input_text = "paraphrase: " + prompt_text.strip()
    inputs = par_tok([input_text], return_tensors="pt", truncation=True, padding=True).to(DEVICE)
    with torch.no_grad():
        out = par_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
            eos_token_id=par_tok.eos_token_id,
            pad_token_id=par_tok.pad_token_id
        )
    decs = [par_tok.decode(o, skip_special_tokens=True, clean_up_tokenization_spaces=True) for o in out]
    # post-process: strip
    decs = [d.strip() for d in decs]
    return decs

def perplexity_score(text):
    """
    Compute (approximate) perplexity of 'text' under GPT-2.
    Lower perplexity -> text more likely under GPT-2 distribution.
    We return the avg negative log-likelihood per token (i.e., cross-entropy) which is monotonic with perplexity.
    """
    enc = gpt2_tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gpt2(**enc, labels=enc["input_ids"])
        # loss is the mean cross-entropy per token
        loss = outputs.loss.item()
    return loss  # lower is 'more model-like' under GPT-2

def sbert_sim(a, b):
    v = sbert.encode([a, b], convert_to_tensor=True, show_progress_bar=False)
    a_v, b_v = v[0], v[1]
    sim = torch.nn.functional.cosine_similarity(a_v, b_v, dim=0).item()
    return sim

# -------- Main generation loop ----------
def main():
    src_path = Path(INPUT_JSONL)
    if not src_path.exists():
        print(f"Input file not found: {INPUT_JSONL}")
        return

    sources = load_sources(INPUT_JSONL)
    out_records = []

    print(f"Processing {len(sources)} sources...")

    for rec in tqdm(sources, desc="sources"):
        src_text = rec.get("source_text") or rec.get("text") or rec.get("prompt") or ""
        if not src_text.strip():
            continue

        item = {
            "id": rec.get("id", str(uuid.uuid4())),
            "number": rec.get("number"),
            "domain": rec.get("domain"),
            "source_text": src_text,
            "length_tokens": len(src_text.split()),
            "timestamp_utc": datetime.utcnow().isoformat() + "Z",
            # fields to be filled
            "simple_paraphrase": None,
            "paraphrase_simple_sim": None,
            "adversarial_paraphrase": None,
            "paraphrase_adv_sim": None,
            "detector_score_adv": None,
            "generation_params": {
                "paraphrase_model": PARAPHRASE_MODEL,
                "temperature": PARAPHRASE_TEMPERATURE,
                "top_p": PARAPHRASE_TOP_P,
                "max_new_tokens": PARAPHRASE_MAX_NEW_TOKENS,
                "num_candidates": PARAPHRASE_NUM_CANDIDATES
            }
        }

        # 1) Simple paraphrase (single)
        try:
            simples = generate_paraphrases(
                src_text,
                num_return_sequences=PARAPHRASE_NUM_SIMPLE,
                max_new_tokens=PARAPHRASE_MAX_NEW_TOKENS,
                temperature=PARAPHRASE_TEMPERATURE,
                top_p=PARAPHRASE_TOP_P
            )
            simple = simples[0] if simples else ""
            sim_s = sbert_sim(src_text, simple)
            item["simple_paraphrase"] = simple
            item["paraphrase_simple_sim"] = round(sim_s, 4)
        except Exception as e:
            item["simple_paraphrase"] = ""
            item["paraphrase_simple_sim"] = None
            print("Simple paraphrase error for id", item["id"], ":", e)

        # 2) Adversarial candidates
        candidates = []
        try:
            # generate in smaller batches to avoid memory blowups if PARAPHRASE_NUM_CANDIDATES large
            batch = 5
            needed = PARAPHRASE_NUM_CANDIDATES
            while len(candidates) < PARAPHRASE_NUM_CANDIDATES:
                nret = min(batch, PARAPHRASE_NUM_CANDIDATES - len(candidates))
                cand = generate_paraphrases(
                    src_text,
                    num_return_sequences=nret,
                    max_new_tokens=PARAPHRASE_MAX_NEW_TOKENS,
                    temperature=max(0.9, PARAPHRASE_TEMPERATURE),  # slightly more diverse for candidates
                    top_p=0.98
                )
                # ensure uniqueness
                for c in cand:
                    c_strip = c.strip()
                    if c_strip and c_strip not in candidates and c_strip != item["simple_paraphrase"] and c_strip != src_text:
                        candidates.append(c_strip)
        except Exception as e:
            print("Candidate generation error for id", item["id"], ":", e)

        # 3) Filter by semantic similarity threshold and compute detector scores
        scored = []
        for c in candidates:
            sim = sbert_sim(src_text, c)
            if sim < SIM_THRESHOLD:
                continue
            # detector proxy: GPT-2 cross-entropy loss (lower => more model-like under GPT-2)
            try:
                det_score = perplexity_score(c)
            except Exception as e:
                print("Perplexity error:", e)
                det_score = None
            scored.append({"text": c, "sim": sim, "detector": det_score})

        # If none passed sim threshold, relax threshold progressively down to 0.65 (but keep track)
        relax = 0
        sim_thr = SIM_THRESHOLD
        while not scored and sim_thr > 0.65 and relax < 3:
            sim_thr -= 0.02
            relax += 1
            for c in candidates:
                sim = sbert_sim(src_text, c)
                if sim < sim_thr:
                    continue
                try:
                    det_score = perplexity_score(c)
                except Exception as e:
                    det_score = None
                scored.append({"text": c, "sim": sim, "detector": det_score})

        # Selection rule:
        # We choose the candidate with the LOWEST detector loss (i.e., most model-like under GPT-2)
        # because such candidates often transfer as stronger paraphrase attacks (heuristic).
        if scored:
            # remove items with None detector score, prefer those with numeric score
            scored_numeric = [s for s in scored if s["detector"] is not None]
            if scored_numeric:
                best = min(scored_numeric, key=lambda x: x["detector"])
            else:
                best = scored[0]
            item["adversarial_paraphrase"] = best["text"]
            item["paraphrase_adv_sim"] = round(best["sim"], 4)
            item["detector_score_adv"] = None if best["detector"] is None else round(best["detector"], 6)
        else:
            item["adversarial_paraphrase"] = ""
            item["paraphrase_adv_sim"] = None
            item["detector_score_adv"] = None

        out_records.append(item)

    # Write out
    write_jsonl(OUTPUT_JSONL, out_records)
    print(f"Wrote {len(out_records)} records to {OUTPUT_JSONL}")

if __name__ == "__main__":
    main()


In [23]:
import json

input_path = "attack_variants_200.jsonl"
output_path = "attack_variants_200.json"

with open(input_path, "r", encoding="utf8") as fin:
    records = [json.loads(line) for line in fin if line.strip()]

with open(output_path, "w", encoding="utf8") as fout:
    json.dump(records, fout, ensure_ascii=False, indent=2)

print(f"Converted {input_path} to {output_path} with {len(records)} records.")

Converted attack_variants_200.jsonl to attack_variants_200.json with 200 records.


#### res.py

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

json_file_path = "C:\\Users\\tenys\\defense_project\\datasets\\eval_outputs_20251007_170127.jsonl"
def analyze_llm_results(json_file_path):
    """
    Analyzes LLM experiment results from a JSON file, aggregates data,
    prints a summary table, and generates Matplotlib plots.

    Args:
        json_file_path (str): The path to the JSON file containing the results.
    """
    try:
        with open(json_file_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{json_file_path}' was not found.")
        return
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{json_file_path}'. Check file format.")
        return

    # Prepare data for DataFrame
    processed_data = []
    for prompt_id, prompt_data in data.items():
        domain = prompt_data.get('domain', 'Unknown')
        for variant_name, metrics in prompt_data['variants'].items():
            processed_data.append({
                'prompt_id': prompt_id,
                'domain': domain,
                'variant': variant_name.capitalize(), # Capitalize for consistent display (Original, Simple, Adversarial)
                'runtime': metrics.get('runtime_s', 0.0),
                'length': metrics.get('response_length', 0),
                'detector_score': metrics.get('detector_score', 0.0),
                'unsafe_label': metrics.get('unsafe_label', False)
            })

    df = pd.DataFrame(processed_data)

    # --- Aggregation ---
    aggregated_df = df.groupby(['domain', 'variant']).agg(
        mean_runtime=('runtime', 'mean'),
        mean_length=('length', 'mean'),
        mean_detector_score=('detector_score', 'mean')
    ).reset_index()

    # --- Print Tabulated Results ---
    print("### Overall Results Aggregated by Prompt Domain and Variant 📊")
    print("\nThis table shows the average (mean) values for runtime, response length, and detector score,")
    print("grouped first by the prompt's domain and then by its variant.")
    print("\n" + "="*80) # Separator for readability
    print(aggregated_df.to_string(index=False))
    print("="*80 + "\n") # Separator for readability

    # --- Matplotlib Visualizations ---

    # Ensure consistent order for variants in plots
    variant_order = ['Original', 'Simple', 'Adversarial']
    aggregated_df['variant'] = pd.Categorical(aggregated_df['variant'], categories=variant_order, ordered=True)
    aggregated_df = aggregated_df.sort_values(['domain', 'variant'])

    # Get unique domains for plotting
    domains = aggregated_df['domain'].unique()
    num_domains = len(domains)
    colors = plt.cm.get_cmap('Paired', num_domains * len(variant_order)) # More distinct colors

    # Plot 1: Mean Detector Score
    fig1, ax1 = plt.subplots(figsize=(12, 7))
    width = 0.25 # width of bars

    x = np.arange(len(domains)) # the label locations for domains

    for i, variant in enumerate(variant_order):
        variant_data = aggregated_df[aggregated_df['variant'] == variant]
        # Align bars for each domain group
        bar_positions = [j + i * width - width for j in x]
        bars = ax1.bar(bar_positions, variant_data['mean_detector_score'], width, label=variant)
        # Add labels on bars
        for bar in bars:
            yval = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.01, round(yval, 2), ha='center', va='bottom', fontsize=8)


    ax1.set_xlabel('Prompt Domain')
    ax1.set_ylabel('Mean Detector Score')
    ax1.set_title('Mean Detector Score by Domain and Variant')
    ax1.set_xticks(x)
    ax1.set_xticklabels(domains, rotation=45, ha='right')
    ax1.legend(title='Variant')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    # Save figure instead of showing it
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = os.getcwd()
    image_path = os.path.join(script_dir, "mean_detector_score.png")
    fig1.savefig(image_path, dpi=300, bbox_inches='tight')
    plt.close(fig1)
    print(f"Saved 'Mean Detector Score' plot to: {image_path}")